# Semana 1 — Listas, diccionarios y conjuntos aplicado a Penonomé

Aplica las unidades 10 a 16 del curso (listas y tuplas, diccionarios y JSON, iteración y búsqueda, listas bidimensionales, CRUD de diccionarios, diccionarios avanzados y conjuntos) a los datos oficiales del Censo 2023 y de la Encuesta de Mercado Laboral, sin pandas.

[Proyecto](../../README.md) | [Anterior](../../README.md) | [Siguiente](s02-funciones-clausuras-clases.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## 0. Leer los CSV con el módulo `csv`

Los datos viven en `datos/penonome/`. Se leen con `open` dentro de un bloque `with` (unidad 13, [lectura y escritura de archivos](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u13-listas-bidimensionales.ipynb)) y con `csv.DictReader`, que entrega cada fila como un diccionario cuyas claves son los encabezados. El resultado es una **lista de diccionarios**.

**Clave:** los CSV del paquete tienen BOM (marca de orden de bytes); por eso se abren con `encoding="utf-8-sig"`. Con `utf-8` la primera clave quedaría como `"\ufeffid_corregimiento"`.

El cargador completo está en `src/penonome/carga.py`; la celda siguiente empieza por su idea central.

Los cargadores de `penonome.carga` hacen lo mismo y, además, convierten las columnas numéricas con `int` o `float`. Una celda vacía (población de 2010 de un corregimiento que aún no existía) queda como `None`.

In [2]:
import csv

from penonome import PAQUETE

with open(PAQUETE / "00_catalogo_corregimientos.csv", encoding="utf-8-sig", newline="") as archivo:
    catalogo = list(csv.DictReader(archivo))

print(type(catalogo), type(catalogo[0]))
print(catalogo[0])
verificar("el catálogo tiene 16 corregimientos", len(catalogo), 16)

from penonome.carga import PADRES_LEY_296, cargar_condicion_actividad, cargar_poblacion, cargar_viviendas

poblacion = cargar_poblacion()                      # tabla 01
viviendas = cargar_viviendas("corregimiento")       # tabla 02, solo corregimientos
distrito = cargar_viviendas("distrito")[0]          # fila total de Penonomé
provincia = cargar_viviendas("provincia")[0]        # fila total de Coclé

print(poblacion[3])
verificar("Coclé tiene 77,384 viviendas en el Censo 2023", provincia["viviendas"], 77384)

<class 'list'> <class 'dict'>
{'id_corregimiento': 'penonome_cabecera', 'provincia': 'Coclé', 'distrito': 'Penonomé', 'corregimiento': 'Penonomé (cabecera)', 'nota_cambio_limites': ''}
Correcto: el catálogo tiene 16 corregimientos
{'id_corregimiento': 'chiguiri_arriba', 'provincia': 'Coclé', 'distrito': 'Penonomé', 'corregimiento': 'Chiguirí Arriba', 'superficie_km2': 104, 'pob_2000': 8581, 'pob_2010': 10018, 'pob_2023': 7185, 'densidad_2023': 69.1, 'nota': 'dividido por la Ley 296 de 26-abr-2022 (corregimientos nuevos)'}
Correcto: Coclé tiene 77,384 viviendas en el Censo 2023


## 1. Listas y tuplas: población 2023

Una lista guarda una secuencia ordenada y modificable; una tupla, una secuencia inmutable ([unidad 10](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u10-listas-y-tuplas.ipynb)). Aquí cada corregimiento se resume en una tupla `(id, nombre, población 2023)`, porque esos tres datos no deben cambiar, y las tuplas se agrupan en una lista.

`sorted` con `reverse=True` ordena de mayor a menor y el rebanado `[:3]` toma los tres primeros. El operador `in` comprueba la pertenencia.

In [3]:
registros = [(fila["id_corregimiento"], fila["corregimiento"], fila["pob_2023"]) for fila in poblacion]

total_2023 = sum(habitantes for _, _, habitantes in registros)
print("Corregimientos:", len(registros))
print("Primero:", registros[0])
print("Último:", registros[-1])
print(f"Población del distrito en 2023: {total_2023:,}")

verificar("los 16 corregimientos suman 104,326 habitantes", total_2023, 104326)
verificar("la suma coincide con la fila distrito de la tabla 02", total_2023, distrito["poblacion"])

por_poblacion = sorted(registros, key=lambda registro: registro[2], reverse=True)
tres_mayores = [nombre for _, nombre, _ in por_poblacion[:3]]
print("Los tres corregimientos más poblados:", tres_mayores)

nombres = [nombre for _, nombre, _ in registros]
print("¿Está 'Riecito' en la lista?", "Riecito" in nombres)

cabecera = registros[0]
try:
    cabecera[2] = 0
except TypeError as error:
    print("Una tupla no se modifica:", error)

verificar("tres más poblados", tres_mayores, ["Penonomé (cabecera)", "El Coco", "Cañaveral"])

Corregimientos: 16
Primero: ('penonome_cabecera', 'Penonomé (cabecera)', 25836)
Último: ('san_miguel', 'San Miguel', 2038)
Población del distrito en 2023: 104,326
Correcto: los 16 corregimientos suman 104,326 habitantes
Correcto: la suma coincide con la fila distrito de la tabla 02
Los tres corregimientos más poblados: ['Penonomé (cabecera)', 'El Coco', 'Cañaveral']
¿Está 'Riecito' en la lista? True
Una tupla no se modifica: 'tuple' object does not support item assignment
Correcto: tres más poblados


## 2. Iteración y búsqueda: ¿qué corregimiento creció más entre 2010 y 2023?

La [unidad 12](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u12-iteracion-y-busqueda.ipynb) recorre secuencias con `for` y busca valores. Antes de comparar hay que filtrar: según la Ley 296 de 2022, cinco corregimientos se dividieron y seis son nuevos, así que solo los que no cambiaron de límites tienen una población de 2010 comparable con la de 2023.

**Error común:** calcular el crecimiento de Pajonal sin más da -43.4 %, pero esa caída no es despoblación: en 2022 se le separó General Victoriano Lorenzo.

In [4]:
divididos = list(PADRES_LEY_296)                         # los cinco que perdieron territorio
nuevos = [hijo for hijos in PADRES_LEY_296.values() for hijo in hijos]

crecimiento = []                                         # lista de tuplas (nombre, %)
for fila in poblacion:
    clave = fila["id_corregimiento"]
    if clave in divididos or clave in nuevos:
        continue
    porcentaje = round((fila["pob_2023"] / fila["pob_2010"] - 1) * 100, 1)
    crecimiento.append((fila["corregimiento"], porcentaje))

for nombre, porcentaje in crecimiento:
    print(f"{nombre:<22}{porcentaje:>7.1f} %")

Penonomé (cabecera)      18.8 %
Cañaveral                24.0 %
Coclé                    41.7 %
El Coco                  97.4 %
Río Grande               12.4 %


Búsqueda lineal del máximo, sin `max`: se guarda el mejor candidato y se compara con cada elemento. Luego se confirma con `max` y una función clave.

Una búsqueda lineal también sirve para encontrar una fila por su nombre. Si no la encuentra, devuelve `None` en lugar de fallar.

In [5]:
mayor = crecimiento[0]
for candidato in crecimiento[1:]:
    if candidato[1] > mayor[1]:
        mayor = candidato

print("Mayor crecimiento (búsqueda lineal):", mayor)
print("Mayor crecimiento (max):", max(crecimiento, key=lambda par: par[1]))

verificar("El Coco casi duplicó su población (97.4 %)", mayor, ("El Coco", 97.4))

def buscar_por_nombre(filas, nombre):
    """Recorre las filas y devuelve la primera cuyo corregimiento coincide, o None."""
    for fila in filas:
        if fila["corregimiento"] == nombre:
            return fila
    return None

riecito = buscar_por_nombre(poblacion, "Riecito")
print(riecito["corregimiento"], riecito["pob_2010"], riecito["pob_2023"])
print(buscar_por_nombre(poblacion, "Antón"))

verificar("Riecito no tiene población de 2010 (no existía)", riecito["pob_2010"], None)

Mayor crecimiento (búsqueda lineal): ('El Coco', 97.4)
Mayor crecimiento (max): ('El Coco', 97.4)
Correcto: El Coco casi duplicó su población (97.4 %)
Riecito None 1315
None
Correcto: Riecito no tiene población de 2010 (no existía)


## 3. Listas bidimensionales: tabla de acceso a internet

Una lista bidimensional es una lista de filas; `tabla[i][j]` lee la fila `i`, columna `j` ([unidad 13](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u13-listas-bidimensionales.ipynb)). Se arma con una comprensión de listas a partir de la tabla 02: viviendas, viviendas sin internet y su porcentaje.

Con un doble bucle se suman las columnas numéricas (1 y 2). La suma debe coincidir con la fila del distrito: es la misma suma de control que usa el paquete.

In [6]:
tabla = [
    [fila["area"], fila["viviendas"], fila["sin_internet"], round(fila["sin_internet"] / fila["viviendas"] * 100, 1)]
    for fila in viviendas
]

print(f"{'Corregimiento':<28}{'Viviendas':>10}{'Sin internet':>14}{'%':>7}")
for nombre, total, sin_internet, porcentaje in tabla:
    print(f"{nombre:<28}{total:>10,}{sin_internet:>14,}{porcentaje:>7.1f}")

totales = [0, 0]
for fila in tabla:
    for j in range(1, 3):
        totales[j - 1] += fila[j]

porcentaje_distrito = round(totales[1] / totales[0] * 100, 1)
print("Viviendas:", totales[0], "| Sin internet:", totales[1], "|", porcentaje_distrito, "%")

verificar("las viviendas de los corregimientos suman 28,547", totales[0], distrito["viviendas"])
verificar("el 41.9 % de las viviendas del distrito no tiene internet", porcentaje_distrito, 41.9)
verificar("celda tabla[7][3]: Río Indio, 99.0 % sin internet", tabla[7][3], 99.0)

Corregimiento                Viviendas  Sin internet      %
Penonomé (cabecera)              7,547         1,323   17.5
Cañaveral                        2,624           828   31.6
Coclé                            1,722           451   26.2
Chiguirí Arriba                  1,712         1,319   77.0
El Coco                          3,271           636   19.4
Pajonal                          1,935         1,115   57.6
Río Grande                       1,090           383   35.1
Río Indio                          953           943   99.0
Toabré                           2,051         1,317   64.2
Tulú                               508           505   99.4
Boca de Tucué                      518           511   98.6
Candelario Ovalle                  758           593   78.2
General Victoriano Lorenzo       2,369           761   32.1
Las Minas                          733           545   74.4
Riecito                            281           276   98.2
San Miguel                         475  

## 4. Diccionarios y JSON

Un diccionario guarda pares clave-valor y permite leer un dato por su clave sin recorrer la lista ([unidad 11](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u11-diccionarios-y-json.ipynb)). La clave natural del paquete es `id_corregimiento`, que une las tablas 01 y 02.

In [7]:
corregimientos = {}
for fila in poblacion:
    corregimientos[fila["id_corregimiento"]] = {
        "nombre": fila["corregimiento"],
        "superficie_km2": fila["superficie_km2"],
        "pob_2010": fila["pob_2010"],
        "pob_2023": fila["pob_2023"],
    }

for fila in viviendas:                       # une la tabla 02 por la misma clave
    datos = corregimientos[fila["id_corregimiento"]]
    datos["viviendas"] = fila["viviendas"]
    datos["sin_internet"] = fila["sin_internet"]

print(corregimientos["el_coco"])
print("{0}: {1:,} habitantes en {2:.1f} km²".format(
    corregimientos["el_coco"]["nombre"], corregimientos["el_coco"]["pob_2023"], corregimientos["el_coco"]["superficie_km2"]))

verificar("El Coco tiene 3,271 viviendas", corregimientos["el_coco"]["viviendas"], 3271)

{'nombre': 'El Coco', 'superficie_km2': 148.3, 'pob_2010': 5605, 'pob_2023': 11064, 'viviendas': 3271, 'sin_internet': 636}
El Coco: 11,064 habitantes en 148.3 km²
Correcto: El Coco tiene 3,271 viviendas


`json.dumps` convierte el diccionario en texto JSON y `json.loads` lo devuelve a diccionario. Se escribe en un archivo temporal con `json.dump` para no dejar archivos nuevos en el proyecto.

**Error común:** JSON no tiene `None`; lo escribe como `null`, y al leerlo vuelve a ser `None`.

In [8]:
import json
import tempfile
from pathlib import Path

texto = json.dumps(corregimientos["riecito"], ensure_ascii=False)
print(texto)

with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "corregimientos.json"
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(corregimientos, archivo, ensure_ascii=False, indent=2)
    with open(ruta, encoding="utf-8") as archivo:
        leidos = json.load(archivo)

verificar("el JSON leído es igual al diccionario original", leidos, corregimientos)
verificar("None se guarda como null", '"pob_2010": null' in texto, True)

{"nombre": "Riecito", "superficie_km2": 83.3, "pob_2010": null, "pob_2023": 1315, "viviendas": 281, "sin_internet": 276}
Correcto: el JSON leído es igual al diccionario original
Correcto: None se guarda como null


## 5. CRUD sobre el diccionario de corregimientos

La [unidad 14](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u14-diccionarios-crud.ipynb) trabaja las cuatro operaciones: crear, leer, actualizar y borrar. Se aplican sobre una copia para no alterar el diccionario de la sección anterior.

**Referencia:** la fila `nivel = distrito` de la tabla 02 es el total de Penonomé; no debe sumarse junto con los corregimientos. El ejemplo la agrega y la retira para mostrar por qué.

In [9]:
import copy

registro = copy.deepcopy(corregimientos)

# Crear: se agrega el total del distrito como si fuera un corregimiento más
registro["total_distrito"] = {"nombre": "Penonomé (distrito)", "pob_2023": distrito["poblacion"]}
print("Con el total, la suma da el doble:", sum(d["pob_2023"] for d in registro.values()))

# Leer: get devuelve un valor por defecto si la clave no existe
print("Superficie de El Coco (km²):", registro.get("el_coco", {}).get("superficie_km2"))
print("Antón:", registro.get("anton", "no es un corregimiento de Penonomé"))

# Actualizar: update agrega la densidad redondeada a la cabecera
cabecera = registro["penonome_cabecera"]
cabecera.update({"densidad": round(cabecera["pob_2023"] / cabecera["superficie_km2"], 1)})
print("Cabecera:", cabecera)

# Borrar: pop retira el total y lo devuelve
retirado = registro.pop("total_distrito")
print("Retirado:", retirado["nombre"])

verificar("con el total incluido la suma se duplica", retirado["pob_2023"] * 2, 208652)
verificar("después de pop quedan 16 corregimientos", len(registro), 16)
verificar("el original no cambió (copia profunda)", "densidad" in corregimientos["penonome_cabecera"], False)

Con el total, la suma da el doble: 208652
Superficie de El Coco (km²): 148.3
Antón: no es un corregimiento de Penonomé
Cabecera: {'nombre': 'Penonomé (cabecera)', 'superficie_km2': 53.4, 'pob_2010': 21748, 'pob_2023': 25836, 'viviendas': 7547, 'sin_internet': 1323, 'densidad': 483.8}
Retirado: Penonomé (distrito)
Correcto: con el total incluido la suma se duplica
Correcto: después de pop quedan 16 corregimientos
Correcto: el original no cambió (copia profunda)


`setdefault` sirve para contar: crea la clave con 0 la primera vez y luego la incrementa. Aquí se cuenta cuántos corregimientos tiene cada tipo de nota en el catálogo.

In [10]:
tipos = {}
for fila in catalogo:
    nota = fila["nota_cambio_limites"]
    if nota.startswith("creado"):
        tipo = "nuevo"
    elif nota.startswith("dividido"):
        tipo = "dividido"
    else:
        tipo = "sin cambios"
    tipos.setdefault(tipo, 0)
    tipos[tipo] += 1

print(tipos)
verificar("6 nuevos, 5 divididos y 5 sin cambios", tipos, {"sin cambios": 5, "dividido": 5, "nuevo": 6})

{'sin cambios': 5, 'dividido': 5, 'nuevo': 6}
Correcto: 6 nuevos, 5 divididos y 5 sin cambios


## 6. Diccionarios avanzados: comprensión, `defaultdict` y diccionario doble

La [unidad 15](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u15-diccionarios-avanzados.ipynb) añade formas compactas de crear y agrupar. Una **comprensión de diccionario** calcula el porcentaje de viviendas sin internet de cada corregimiento en una sola expresión.

In [11]:
sin_internet = {
    fila["id_corregimiento"]: round(fila["sin_internet"] / fila["viviendas"] * 100, 1)
    for fila in viviendas
}
mayoria_sin_internet = {clave: pct for clave, pct in sin_internet.items() if pct > 90}

print("Más del 90 % de viviendas sin internet:", mayoria_sin_internet)
verificar("cinco corregimientos superan el 90 %", len(mayoria_sin_internet), 5)
verificar("Tulú es el más desconectado (99.4 %)", max(sin_internet, key=sin_internet.get), "tulu")

Más del 90 % de viviendas sin internet: {'rio_indio': 99.0, 'tulu': 99.4, 'boca_de_tucue': 98.6, 'riecito': 98.2, 'san_miguel': 94.9}
Correcto: cinco corregimientos superan el 90 %
Correcto: Tulú es el más desconectado (99.4 %)


`defaultdict(list)` agrupa sin preguntar si la clave existe. Se agrupan los corregimientos nuevos bajo su corregimiento original, y con ese grupo se reconstruye el territorio de 2010: así el crecimiento sí es comparable.

In [12]:
from collections import defaultdict

poblacion_por_id = {fila["id_corregimiento"]: fila for fila in poblacion}

territorio_2010 = defaultdict(list)
for padre, hijos in PADRES_LEY_296.items():
    territorio_2010[padre].append(padre)
    territorio_2010[padre].extend(hijos)

for padre, partes in territorio_2010.items():
    antes = poblacion_por_id[padre]["pob_2010"]
    ahora = sum(poblacion_por_id[parte]["pob_2023"] for parte in partes)
    print(f"{poblacion_por_id[padre]['corregimiento']:<16} 2010: {antes:>6,}  2023: {ahora:>6,}  {(ahora / antes - 1) * 100:>5.1f} %")

pajonal_2023 = sum(poblacion_por_id[parte]["pob_2023"] for parte in territorio_2010["pajonal"])
verificar("Pajonal con General Victoriano Lorenzo suma 16,174 en 2023", pajonal_2023, 16174)

Chiguirí Arriba  2010: 10,018  2023: 10,800    7.8 %
Pajonal          2010: 13,565  2023: 16,174   19.2 %
Río Indio        2010:  5,240  2023:  5,341    1.9 %
Toabré           2010: 10,203  2023: 11,568   13.4 %
Tulú             2010:  4,624  2023:  4,903    6.0 %
Correcto: Pajonal con General Victoriano Lorenzo suma 16,174 en 2023


Un **diccionario doble** (un diccionario dentro de otro) ordena la tasa de desempleo de la EML por área y por fecha: `tasas[area][periodo]`.

**Referencia:** la tasa de la EML se mide distinto de la tasa censal; se compara el orden de magnitud, no el número exacto (LEEME del paquete, sección 1).

In [13]:
condicion = cargar_condicion_actividad()

tasas = {}
for fila in condicion:
    if fila["indicador"] == "Tasa de desempleo (%)":
        tasas[fila["area"]] = {"oct2024": fila["oct2024_total"], "sep2025": fila["sep2025_total"]}

for area, por_fecha in tasas.items():
    print(f"{area:<14} oct-2024: {por_fecha['oct2024']:>4} %   sep-2025: {por_fecha['sep2025']:>4} %")

verificar("Coclé, desempleo de sep-2025", tasas["Coclé"]["sep2025"], 7.1)
verificar("Coclé urbana subió de 6.3 % a 9.1 %", (tasas["Coclé urbana"]["oct2024"], tasas["Coclé urbana"]["sep2025"]), (6.3, 9.1))

República      oct-2024:  9.7 %   sep-2025: 10.4 %
Coclé          oct-2024:  5.7 %   sep-2025:  7.1 %
Coclé urbana   oct-2024:  6.3 %   sep-2025:  9.1 %
Coclé rural    oct-2024:  5.2 %   sep-2025:  5.8 %
Correcto: Coclé, desempleo de sep-2025
Correcto: Coclé urbana subió de 6.3 % a 9.1 %


## 7. Conjuntos: corregimientos nuevos e indicadores por año

Un conjunto guarda elementos sin repetir y sin orden, y permite unión `|`, intersección `&`, diferencia `-` y diferencia simétrica `^` ([unidad 16](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u16-conjuntos.ipynb)). Primero, los corregimientos: los nuevos son los que no tienen población en 2010.

Con el conjunto de afectados se calcula el crecimiento del bloque de once corregimientos que el LEEME recomienda comparar como uno solo.

In [14]:
todos = {fila["id_corregimiento"] for fila in poblacion}
nuevos = {fila["id_corregimiento"] for fila in poblacion if fila["pob_2010"] is None}
padres = set(PADRES_LEY_296)
sin_cambios = todos - nuevos - padres
afectados = padres | nuevos

print("Nuevos:", sorted(nuevos))
print("Sin cambios de límites:", sorted(sin_cambios))
print("¿Los nuevos y los padres son disjuntos?", nuevos.isdisjoint(padres))
print("¿Los afectados son subconjunto de todos?", afectados <= todos)

verificar("los nuevos coinciden con los hijos que nombra la Ley 296",
          nuevos, {hijo for hijos in PADRES_LEY_296.values() for hijo in hijos})
verificar("cinco corregimientos no cambiaron de límites", len(sin_cambios), 5)

bloque_2010 = sum(poblacion_por_id[clave]["pob_2010"] or 0 for clave in afectados)
bloque_2023 = sum(poblacion_por_id[clave]["pob_2023"] for clave in afectados)
crecimiento_bloque = round((bloque_2023 / bloque_2010 - 1) * 100, 1)

crecimiento_distrito = round((total_2023 / sum(f["pob_2010"] for f in poblacion if f["pob_2010"]) - 1) * 100, 1)
print(f"Bloque de los once: {bloque_2010:,} -> {bloque_2023:,} ({crecimiento_bloque} %)")
print(f"Distrito completo: {crecimiento_distrito} %")

verificar("el bloque de once creció 11.8 %", crecimiento_bloque, 11.8)
verificar("el distrito creció 21.7 % entre 2010 y 2023", crecimiento_distrito, 21.7)

Nuevos: ['boca_de_tucue', 'candelario_ovalle', 'general_victoriano_lorenzo', 'las_minas', 'riecito', 'san_miguel']
Sin cambios de límites: ['canaveral', 'cocle', 'el_coco', 'penonome_cabecera', 'rio_grande']
¿Los nuevos y los padres son disjuntos? True
¿Los afectados son subconjunto de todos? True
Correcto: los nuevos coinciden con los hijos que nombra la Ley 296
Correcto: cinco corregimientos no cambiaron de límites
Bloque de los once: 43,650 -> 48,786 (11.8 %)
Distrito completo: 21.7 %
Correcto: el bloque de once creció 11.8 %
Correcto: el distrito creció 21.7 % entre 2010 y 2023


Segundo uso: ¿qué indicadores de la base larga (tabla 99) existen en cada año? El nombre del indicador antes de ` | ` es el indicador base. La intersección dice qué se puede seguir en el tiempo; la diferencia, qué aparece solo en un año.

In [15]:
from penonome.carga import cargar_base_larga

base = cargar_base_larga()

def indicadores_de(fecha):
    """Conjunto de indicadores base (sin desagregación) con dato en esa fecha."""
    return {fila["indicador"].split(" | ")[0] for fila in base if fila["fecha"] == fecha}

css_2016, css_2020, css_2024 = indicadores_de("2016"), indicadores_de("2020"), indicadores_de("2024")
comunes = css_2016 & css_2020 & css_2024

print("En los tres años:", sorted(comunes))
print("Solo en 2020 respecto de 2016:", sorted(css_2020 - css_2016))
print("Población en los tres censos:", indicadores_de("2000") & indicadores_de("2010") & indicadores_de("2023"))

verificar("cinco indicadores de la CSS se siguen en 2016, 2020 y 2024", len(comunes), 5)
verificar("entre censos solo la población es comparable",
          indicadores_de("2000") & indicadores_de("2010") & indicadores_de("2023"), {"poblacion"})

En los tres años: ['css_asegurados_activos', 'css_asegurados_total', 'css_dependientes', 'css_pensionados', 'css_poblacion_protegida']
Solo en 2020 respecto de 2016: ['pib_volumen']
Población en los tres censos: {'poblacion'}
Correcto: cinco indicadores de la CSS se siguen en 2016, 2020 y 2024
Correcto: entre censos solo la población es comparable


`zip` une dos secuencias elemento a elemento. Se usa para emparejar el porcentaje sin internet con la tasa de desocupación censal de cada corregimiento y ver si van juntos.

**Ampliación (no está en el PDF):** la tasa censal es `desocupados / (ocupados + desocupados)`; en zonas rurales con mucho trabajo agrícola por cuenta propia suele salir muy baja, aunque falten servicios.

In [16]:
claves = [fila["id_corregimiento"] for fila in viviendas]
desocupacion = [round(f["desocupados"] / (f["ocupados"] + f["desocupados"]) * 100, 1) for f in viviendas]
pares = dict(zip(claves, zip([sin_internet[c] for c in claves], desocupacion)))

for clave in ("penonome_cabecera", "pajonal", "tulu", "riecito"):
    print(f"{clave:<20} sin internet {pares[clave][0]:>5} %   desocupación censal {pares[clave][1]:>5} %")

verificar("la tasa censal del distrito es 6.3 %",
          round(distrito["desocupados"] / (distrito["ocupados"] + distrito["desocupados"]) * 100, 1), 6.3)

penonome_cabecera    sin internet  17.5 %   desocupación censal   8.1 %
pajonal              sin internet  57.6 %   desocupación censal  11.5 %
tulu                 sin internet  99.4 %   desocupación censal   0.6 %
riecito              sin internet  98.2 %   desocupación censal   0.6 %
Correcto: la tasa censal del distrito es 6.3 %


## Conclusiones

- Penonomé tenía **104,326 habitantes** en 2023, un **21.7 %** más que en 2010. La cabecera, El Coco y Cañaveral concentran el 44.3 %.
- Entre los corregimientos que no cambiaron de límites, **El Coco** casi duplicó su población (**+97.4 %**). Los cinco que se dividieron por la Ley 296 de 2022 parecen perder población, pero al sumar sus corregimientos nuevos el bloque creció **11.8 %**: la caída aparente es un efecto de los límites, no de la migración.
- El **41.9 %** de las viviendas del distrito no tiene internet (44.0 % en Coclé). La brecha es territorial: 17.5 % en la cabecera frente a más del 90 % en cinco corregimientos rurales, con Tulú en 99.4 %.
- En la EML, el desempleo de Coclé pasó de **5.7 %** (oct-2024) a **7.1 %** (sep-2025), y el alza vino sobre todo del área urbana (6.3 % a 9.1 %).
- La tasa de desocupación censal es más baja donde falta todo (Tulú 0.6 %, Riecito 0.6 %), lo que indica autoempleo agrícola más que pleno empleo.
- Entre censos, el único indicador comparable del paquete es la población; para seguir el empleo en el tiempo hay que usar la EML y la CSS.

[Proyecto](../../README.md) | [Anterior](../../README.md) | [Siguiente](s02-funciones-clausuras-clases.ipynb)